In [1]:
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
DATA_PATH = Path("../data/assurance-maladie.csv")

df = pd.read_csv(DATA_PATH)

df = df.drop_duplicates()

df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
X = df.drop("charges", axis=1)
y = df["charges"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [5]:
numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [6]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "SVR": SVR(),
    "XGBoost": XGBRegressor(random_state=42)
}

In [7]:
results = []

for model_name, model in models.items():

    # Create full pipeline
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    predictions = pipeline.predict(X_test)

    # Metrics
    rmse = mean_squared_error(y_test, predictions) ** 0.5
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    # Save results
    results.append({
        "Model": model_name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

    print(f"\n========== {model_name} ==========")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE : {mae:.2f}")
    print(f"R2  : {r2:.4f}")


========== Linear Regression ==========
RMSE: 5956.34
MAE : 4177.05
R2  : 0.8069

========== Random Forest ==========
RMSE: 4679.23
MAE : 2583.62
R2  : 0.8808

========== SVR ==========
RMSE: 14425.34
MAE : 9260.28
R2  : -0.1324

========== XGBoost ==========
RMSE: 5059.14
MAE : 2857.00
R2  : 0.8607


In [8]:
results_df = pd.DataFrame(results)

results_df.sort_values(by="R2", ascending=False)

,Model,RMSE,MAE,R2
1,Random Forest,4679.234933,2583.622305,0.880846
3,XGBoost,5059.140897,2856.995046,0.860713
0,Linear Regression,5956.342894,4177.045561,0.806929
2,SVR,14425.336625,9260.280186,-0.132427
